# 02 · Spike sorting with SpyKING CIRCUS 2
Groups detected spikes into units (putative neurons). Runs on CPU (no GPU needed).

## 1. Setup
Load the MaxOne test file. `HDF5_PLUGIN_PATH` must be set before importing SpikeInterface so HDF5 can read MaxWell's compressed data.

In [4]:
import os # access environment settings
from pathlib import Path # file paths
os.environ["HDF5_PLUGIN_PATH"] = str(Path.home() / "hdf5_plugin_path_maxwell") # tell HDF5 where MaxWell's decompression plugin is; must run before importing spikeinterface

import numpy as np # arrays and maths
import matplotlib.pyplot as plt # plotting
import spikeinterface as si # core SpikeInterface
import spikeinterface.extractors as se # file readers
import spikeinterface.preprocessing as spre # cleaning tools
import spikeinterface.sorters as ss # spike sorters

file_path = Path("../data/raw/maxone_record_000011.raw.h5") # the test file
rec = se.read_maxwell(file_path) # open the recording
rec_s = spre.unsigned_to_signed(rec) # uint16 -> signed, ready for the sorter
dur = rec.get_total_duration() # length in s, for firing rates
print(rec) # check it loaded
from pathlib import Path # file paths
import numpy as np # arrays and maths
import matplotlib.pyplot as plt # plotting
import spikeinterface as si # core SpikeInterface
import spikeinterface.extractors as se # file readers
import spikeinterface.preprocessing as spre # cleaning tools
import spikeinterface.sorters as ss # spike sorters

file_path = Path("../data/raw/maxone_record_000011.raw.h5") # the test file
rec = se.read_maxwell(file_path) # open the recording
rec_s = spre.unsigned_to_signed(rec) # uint16 -> signed, ready for the sorter
dur = rec.get_total_duration() # length in s, for firing rates
print(rec) # check it loaded

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
MaxwellRecordingExtractor: 25 channels - 20.0kHz - 1 segments - 1,201,200 samples 
                           60.06s (1.00 minutes) - uint16 dtype - 57.28 MiB
  file_path: /Users/arelibalderas/git_repos/hdmea-organoid-pipeline/notebooks/../data/raw/maxone_record_000011.raw.h5
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
MaxwellRecordingExtractor: 25 channels - 20.0kHz - 1 segments - 1,201,200 samples 
                           60.06s (1.00 minutes) - uint16 dtype - 57.28 MiB
  file_path: /Users/arelibalderas/git_repos/hdmea-organoid-pipeline/notebooks/../data/raw/maxone_record_000011.raw.h5


## 2. Sort, save and reload
SC2 does its own filtering, detection, clustering and template matching.
Settings: `deterministic_peaks_detection=True`; motion correction off (70 µm patch is too small to estimate drift).
Sorting is not fully reproducible: core units are stable, borderline units vary between runs. So we sort once (`run_sorting = True`), save, and reload (`run_sorting = False`).
Units and spike counts: see output below. Compare with 01: 4404 electrode-level detections.

In [5]:
si.set_global_job_kwargs(n_jobs=4, progress_bar=True) # back to 4 cores (1 core didn't fix it)
sort_dir = Path("../data/sorting/sc2_test") # where the sorting is saved (inside data/, ignored by Git)

run_sorting = False # True = run a NEW sorting and overwrite the saved one; False = load the saved one
if run_sorting: # only when you ask for it
    sorting = ss.run_sorter(
        "spykingcircus2", # which sorter
        rec_s, # signed, unfiltered recording: SC2 filters it itself
        folder=sort_dir, # where results are written
        remove_existing_folder=True, # overwrite old results
        verbose=True, # print progress
        deterministic_peaks_detection=True, # detect all peaks, no early stopping
        apply_motion_correction=False, # no drift correction on a 70 µm patch
    )
else:
    sorting = ss.read_sorter_folder(sort_dir) # load the saved result: same units every time
print(sorting) # summary: number of units, duration

units = sorting.get_unit_ids() # one ID per neuron ("unit")
print(f"\nUnits found: {len(units)}") # number of neurons
n_total = 0 # running total of spikes
for u in units: # one row per unit
    n = len(sorting.get_unit_spike_train(u)) # spikes of this unit
    n_total += n # add to total
    print(f"Unit {u}: {n} spikes, {n / dur:.2f} Hz") # count and firing rate
print(f"Total sorted spikes: {n_total}") # compare with 01: 4404 electrode-level detections

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
NumpyFolder (NumpyFolderSorting): 12 units - 1 segments - 20.0kHz

Units found: 12
Unit 1: 65 spikes, 1.08 Hz
Unit 3: 94 spikes, 1.57 Hz
Unit 5: 65 spikes, 1.08 Hz
Unit 7: 31 spikes, 0.52 Hz
Unit 12: 56 spikes, 0.93 Hz
Unit 15: 73 spikes, 1.22 Hz
Unit 17: 46 spikes, 0.77 Hz
Unit 18: 127 spikes, 2.11 Hz
Unit 19: 99 spikes, 1.65 Hz
Unit 21: 64 spikes, 1.07 Hz
Unit 23: 60 spikes, 1.00 Hz
Unit 24: 135 spikes, 2.25 Hz
Total sorted spikes: 915
